<a href="https://colab.research.google.com/github/silvia-j-escobar/ExternDataScience/blob/main/Creating_Chatbot_interface.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [58]:
Silvia Escobar

In [61]:
!pip install transformers

In [64]:
import gradio as gr
import pypdf
import time
from transformers import pipeline
import datetime # Import for generating unique filenames

# --- Global Variables Initialization ---
document_content = "" # Initialize global variable for processed PDF content
current_mode = "Question Answering" # Initialize global variable for operational mode

# --- Question-Answering Pipeline Setup ---
# Initialize the question-answering pipeline using a pre-trained model
# 'distilbert-base-cased-distilled-squad' is a smaller, faster model for demonstration purposes
qa_pipeline = pipeline("question-answering", model="distilbert-base-cased-distilled-squad")

# --- PDF Processing Function ---
# This function handles the upload and text extraction from multiple PDF files.
def process_pdf(files):
    global document_content
    document_content = ""
    if files is not None:
        status_message = f"Processing {len(files)} PDFs...\n"
        for file_obj in files:
            try:
                # Use pypdf to read and extract text from each PDF file
                reader = pypdf.PdfReader(file_obj.name)
                file_text = ""
                for page_num in range(len(reader.pages)):
                    file_text += reader.pages[page_num].extract_text() + "\n"
                document_content += file_text + "\n" # Append text to global content
                status_message += f"  - Successfully extracted text from: {file_obj.name}\n"
            except Exception as e:
                # Handle errors during PDF processing for individual files
                print(f"Error processing PDF {file_obj.name}: {e}")
                document_content += f"<Error: Could not process PDF {file_obj.name}>\n"
                status_message += f"  - Error processing {file_obj.name}: {e}\n"
        status_message += f'All {len(files)} PDFs processed successfully!'
        return status_message
    return "No PDFs uploaded. Please upload one or more PDF files."

# --- Chat Handling Function ---
# This function processes user messages and generates responses using the QA pipeline.
def handle_chat(message, history):
    global current_mode
    if not document_content:
        # If no document is processed, prompt the user to upload one
        mock_response = "Please upload and process a PDF document first."
    else:
        try:
            # Use the QA pipeline to answer the question based on the document content
            qa_result = qa_pipeline(question=message, context=document_content)
            mock_response = qa_result['answer']
        except Exception as e:
            # Handle errors during question answering
            mock_response = f"Error getting answer from document: {e}"
    # Simulate some processing time for chat to give a more realistic feel
    time.sleep(1)
    # Correctly format the history for Gradio's Chatbot (type='messages' expects dicts)
    return history + [{'role': 'user', 'content': message}, {'role': 'assistant', 'content': mock_response}]

# --- Operational Mode Change Function ---
# This function updates the global current_mode based on user selection.
def change_mode(mode):
    global current_mode
    current_mode = mode
    return f"Operational mode set to: {mode}"

# --- Chat History Saving Function ---
# This function saves the current chat history to a timestamped text file.
def save_chat_history(history):
    if not history:
        return "No chat history to save."

    # Generate a unique filename using the current timestamp
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"chat_history_{timestamp}.txt"

    # Write each chat message to the file
    with open(filename, "w", encoding="utf-8") as f:
        for chat_message in history:
            f.write(f"{chat_message['role'].capitalize()}: {chat_message['content']}\n")
        f.write("\n--- End of Chat ---\n")

    return f"Chat history saved to {filename}"


# --- Gradio UI Layout Definition ---
with gr.Blocks(title="Step 5: Full Functioning UI") as demo:
    gr.Markdown("### Step 5: Connected UI")

    with gr.Column(scale=2):
        # Chatbot display area
        chatbot = gr.Chatbot(label="Chat History", height=300, type='messages', allow_tags=False)
        # User input textbox
        user_input = gr.Textbox(
            placeholder="Ask a question about your document...",
            label="Your Question"
        )
        # Buttons for sending message and clearing chat
        send_btn = gr.Button("📤 Send")
        clear_btn = gr.Button("🗑️ Clear Chat")
        # Status display for chat operations (e.g., 'Generating response...')
        chat_status = gr.Textbox(label="Chat Status", interactive=False, lines=1)
        # Button to save chat history
        save_chat_btn = gr.Button("💾 Save Chat")

    with gr.Column(scale=1):
        # File upload component for PDFs
        pdf_input = gr.File(label="📄 Upload PDF", file_types=[".pdf"], file_count="multiple")
        # Button to trigger PDF processing
        process_btn = gr.Button("🔄 Process Document")
        # Status display for PDF processing
        processing_status = gr.Textbox(label="Processing Status", interactive=False, lines=5)

        # Dropdown for selecting operational modes
        operational_mode = gr.Dropdown(
            label="Operational Mode",
            choices=["Question Answering", "Summarization", "Extract Key Info"],
            value="Question Answering",
            interactive=True
        )
        # Status display for mode changes
        mode_status = gr.Textbox(label="Mode Status", interactive=False, lines=1)

    # --- Event Handlers ---
    # Link process button to the PDF processing function
    process_btn.click(process_pdf, inputs=pdf_input, outputs=processing_status)

    # Event chain for sending messages from send_btn:
    # 1. Update chat status to 'Generating response...'
    # 2. Call handle_chat function with user input and current chatbot history
    # 3. Clear chat status (after response is generated)
    send_btn.click(lambda: gr.update(value="Generating response..."), None, chat_status, queue=False)
    send_btn.click(handle_chat, inputs=[user_input, chatbot], outputs=chatbot)
    send_btn.click(lambda: gr.update(value=""), None, chat_status, queue=False)

    # Event chain for sending messages by pressing Enter in user_input:
    # (Same logic as send_btn click)
    user_input.submit(lambda: gr.update(value="Generating response..."), None, chat_status, queue=False)
    user_input.submit(handle_chat, inputs=[user_input, chatbot], outputs=chatbot)
    user_input.submit(lambda: gr.update(value=""), None, chat_status, queue=False)

    # Link clear button to clear the chatbot history
    clear_btn.click(lambda: [], outputs=chatbot)

    # Link operational mode dropdown to the change_mode function
    operational_mode.change(change_mode, inputs=operational_mode, outputs=mode_status)

    # Link save chat button to the save_chat_history function
    save_chat_btn.click(save_chat_history, inputs=chatbot, outputs=chat_status)

# --- Launch the Gradio Demo ---
demo.launch()

Device set to use cpu
/tmp/ipython-input-663426959.py:93: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(label="Chat History", height=300, type='messages', allow_tags=False)


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5404dce5e535655925.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
